In [1]:
from pathlib import Path
input_folder  =  Path(r"E:\Aaron\Cong_Speeches\Github_Upload\Technical Validation\Validation_2\Full_Data_Validation")

In [2]:
from pathlib import Path

# Use the third party "regex" module so that \h works as in your pattern.
# Install once if needed:
#   pip install regex
import regex as re

COMMENT_PATTERN = re.compile(r"\h+#[^\r\n]*$", re.MULTILINE)
IMPORT_PATTERN = re.compile(r"^\s*(?:import\s+\S+|from\s+\S+\s+import\s+.+)")

from pathlib import Path
import regex as re  # pip install regex

COMMENT_PATTERN = re.compile(r"\h+#[^\r\n]*$", re.MULTILINE)
IMPORT_PATTERN = re.compile(r"^\s*(?:import\s+\S+|from\s+\S+\s+import\s+.+)")

def remove_comments_using_given_regex(text: str) -> str:
    """
    Remove comments using exactly \h+#[^\r\n]*$ by adding a temporary space
    before every line so that even lines starting with '#' get matched.
    """
    # Add one space before every line
    transformed = " " + text.replace("\n", "\n ")
    # Apply your regex globally
    cleaned = COMMENT_PATTERN.sub("", transformed)
    # Remove the artificial spaces
    if cleaned.startswith(" "):
        cleaned = cleaned[1:]
    cleaned = cleaned.replace("\n ", "\n")
    return cleaned


def reorder_imports_to_top(text: str) -> str:
    lines = text.splitlines()

    imports = []
    seen_imports = set()
    body_lines = []

    for line in lines:
        if IMPORT_PATTERN.match(line):
            stripped_import = line.strip()
            if stripped_import not in seen_imports:
                seen_imports.add(stripped_import)
                imports.append(stripped_import)
        else:
            body_lines.append(line)

    # Remove leading empty lines in the body
    while body_lines and body_lines[0].strip() == "":
        body_lines.pop(0)

    new_lines = []
    if imports:
        new_lines.extend(imports)
        if body_lines:
            new_lines.append("")  # one blank line after imports

    new_lines.extend(body_lines)

    return "\n".join(new_lines) + "\n"

def process_file(path: Path) -> None:
    original = path.read_text(encoding="utf-8")

    without_comments = remove_comments_using_given_regex(original)
    final_text = reorder_imports_to_top(without_comments)

    path.write_text(final_text, encoding="utf-8")
    print(f"Processed {path}")

def main() -> None:
    folder = input_folder

    if not folder.is_dir():
        print('Folder "test" does not exist')
        return

    for py_file in folder.glob("*.py"):
        process_file(py_file)

if __name__ == "__main__":
    main()


Processed E:\Aaron\Cong_Speeches\Github_Upload\Technical Validation\Validation_2\Full_Data_Validation\2-Scores_Sign_correction_FULL.py


## Remove blank lines

In [3]:
import os


def normalize_empty_lines_in_file(file_path: str) -> None:
    """
    Reduces consecutive empty lines to at most one blank line.
    All other content remains unchanged.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    new_lines = []
    previous_was_empty = False

    for line in lines:
        if line.strip() == "":
            # This line is empty (or contains only whitespace)
            if not previous_was_empty:
                new_lines.append("\n")
                previous_was_empty = True
            # If previous was empty, skip this line
        else:
            new_lines.append(line)
            previous_was_empty = False

    with open(file_path, "w", encoding="utf-8") as f:
        f.writelines(new_lines)


def process_folder(folder_path: str) -> None:
    """
    Walks through folder recursively and processes all .py files.
    """
    for root, _, files in os.walk(folder_path):
        for file_name in files:
            if file_name.endswith(".py"):
                file_path = os.path.join(root, file_name)
                normalize_empty_lines_in_file(file_path)


# Run processing on the already defined input_folder
process_folder(input_folder)

## Remove you/your etc

from pathlib import Path
import regex as re  # pip install regex



# pattern, matches whole word "you" or "your", case insensitive
WORD_PATTERN = re.compile(r"\b(you|your)\b", re.IGNORECASE)

def remove_you_and_your(text: str) -> str:
    # replace with empty string, remove leftover double spaces
    cleaned = WORD_PATTERN.sub("", text)
    cleaned = re.sub(r" {2,}", " ", cleaned)  # collapse multiple spaces
    return cleaned

def process_file(path: Path) -> None:
    text = path.read_text(encoding="utf-8")
    cleaned = remove_you_and_your(text)
    path.write_text(cleaned, encoding="utf-8")
    print(f"Processed {path}")

def main() -> None:
    folder = input_folder

    if not folder.is_dir():
        print(f"Folder does not exist: {folder}")
        return

    for py_file in folder.glob("*.py"):
        process_file(py_file)

if __name__ == "__main__":
    main()
